# 🎓 Student Performance Analysis
### Practical: Load External SQL Data in Google Colab + Pandas & NumPy Operations

**Dataset:** `students_data.sql` — 20 student records across 5 courses  
**Tools:** `sqlite3` · `pandas` · `numpy` · `matplotlib`

---


## 📦 Step 1 — Install Libraries & Import Modules

In [ ]:
# Standard libraries (pre-installed in Colab)
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

print("✅ All libraries imported successfully!")
print(f"   pandas  version : {pd.__version__}")
print(f"   numpy   version : {np.__version__}")


## 📂 Step 2 — Upload the SQL File (External Data Loading)

Run the cell below. A **Choose Files** button will appear.  
Upload the `students_data.sql` file you downloaded alongside this notebook.


In [ ]:
from google.colab import files

print("👆 Click 'Choose Files' and upload students_data.sql")
uploaded = files.upload()

sql_filename = list(uploaded.keys())[0]
print(f"\n✅ File uploaded: {sql_filename}  ({len(uploaded[sql_filename])} bytes)")


## 🗄️ Step 3 — Create SQLite Database & Load SQL File

In [ ]:
DB_PATH = "students.db"

# Connect (creates fresh DB in Colab's runtime)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Read and execute the uploaded SQL file
with open(sql_filename, "r") as f:
    sql_script = f.read()

conn.executescript(sql_script)
conn.commit()

# Verify table was created
cursor.execute("SELECT COUNT(*) FROM students")
row_count = cursor.fetchone()[0]
print(f"✅ Database ready!  Table 'students' has {row_count} rows.")


## 🐼 Step 4 — Load SQL Table into a Pandas DataFrame

In [ ]:
df = pd.read_sql_query("SELECT * FROM students", conn)

print("📋 Shape:", df.shape)
print("\n📌 Column Names:", df.columns.tolist())
print()
df.head(10)


## 🔍 Step 5 — Basic DataFrame Inspection

In [ ]:
print("=" * 50)
print("📊 DataFrame Info")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("❓ Missing Values per Column")
print("=" * 50)
print(df.isnull().sum())

print("\n" + "=" * 50)
print("🔢 Data Types")
print("=" * 50)
print(df.dtypes)


## 🛠️ Step 6 — Pandas Operations

In [ ]:
# ── 6a. Add calculated column: Total Marks & Percentage ──────────────────────
df["total_marks"]   = df["marks_math"] + df["marks_science"] + df["marks_english"]
df["percentage"]    = (df["total_marks"] / 300 * 100).round(2)

# ── 6b. Grade assignment ──────────────────────────────────────────────────────
def assign_grade(pct):
    if pct >= 90: return "A+"
    elif pct >= 80: return "A"
    elif pct >= 70: return "B"
    elif pct >= 60: return "C"
    else: return "D"

df["grade"] = df["percentage"].apply(assign_grade)

print("✅ New columns added: total_marks, percentage, grade")
df[["name", "marks_math", "marks_science", "marks_english",
    "total_marks", "percentage", "grade"]].head(10)


In [ ]:
# ── 6c. Filter: Students with percentage >= 85 ────────────────────────────────
top_students = df[df["percentage"] >= 85].sort_values("percentage", ascending=False)
print(f"🏆 Top Students (≥85%) — {len(top_students)} found:\n")
print(top_students[["name", "course", "city", "percentage", "grade"]].to_string(index=False))


In [ ]:
# ── 6d. GroupBy: Average percentage per Course ───────────────────────────────
course_avg = (df.groupby("course")["percentage"]
                .agg(["mean", "min", "max", "count"])
                .rename(columns={"mean":"Avg %","min":"Min %","max":"Max %","count":"Students"})
                .round(2)
                .sort_values("Avg %", ascending=False))

print("📚 Course-wise Performance:\n")
print(course_avg.to_string())


In [ ]:
# ── 6e. GroupBy: Gender-wise fee collection ───────────────────────────────────
gender_fee = df.groupby("gender")["fee_paid"].agg(["sum","mean","count"])
gender_fee.columns = ["Total Fee (₹)", "Avg Fee (₹)", "Students"]
gender_fee["Total Fee (₹)"] = gender_fee["Total Fee (₹)"].apply(lambda x: f"₹{x:,.0f}")
gender_fee["Avg Fee (₹)"]   = gender_fee["Avg Fee (₹)"].apply(lambda x: f"₹{x:,.0f}")
print("💰 Gender-wise Fee Analysis:\n")
print(gender_fee.to_string())


In [ ]:
# ── 6f. Sorting & Selecting top 5 by attendance ──────────────────────────────
print("📅 Top 5 Students by Attendance:\n")
top_attend = df.nlargest(5, "attendance_percent")[["name","course","attendance_percent","percentage"]]
print(top_attend.to_string(index=False))

# ── 6g. Value counts: Grade distribution ─────────────────────────────────────
print("\n📊 Grade Distribution:")
print(df["grade"].value_counts().sort_index())


## 🔢 Step 7 — NumPy Operations

In [ ]:
# Convert marks columns to NumPy arrays
math_arr    = np.array(df["marks_math"])
sci_arr     = np.array(df["marks_science"])
eng_arr     = np.array(df["marks_english"])
attend_arr  = np.array(df["attendance_percent"])
pct_arr     = np.array(df["percentage"])

print("=" * 55)
print("📐 NumPy Statistical Analysis — Marks & Attendance")
print("=" * 55)

for label, arr in [("Math", math_arr), ("Science", sci_arr),
                   ("English", eng_arr), ("Attendance %", attend_arr),
                   ("Overall %", pct_arr)]:
    print(f"\n  {label}:")
    print(f"    Mean     : {np.mean(arr):.2f}")
    print(f"    Median   : {np.median(arr):.2f}")
    print(f"    Std Dev  : {np.std(arr):.2f}")
    print(f"    Variance : {np.var(arr):.2f}")
    print(f"    Min/Max  : {np.min(arr):.1f} / {np.max(arr):.1f}")


In [ ]:
# ── 7b. Correlation matrix using NumPy ───────────────────────────────────────
marks_matrix = np.column_stack([math_arr, sci_arr, eng_arr, pct_arr])
corr_matrix  = np.corrcoef(marks_matrix.T)

labels = ["Math", "Science", "English", "Overall %"]
print("🔗 Correlation Matrix (NumPy):\n")
header = f"{'':>12}" + "".join(f"{l:>12}" for l in labels)
print(header)
for i, row_label in enumerate(labels):
    row = f"{row_label:>12}" + "".join(f"{corr_matrix[i,j]:>12.4f}" for j in range(4))
    print(row)


In [ ]:
# ── 7c. Percentile analysis ───────────────────────────────────────────────────
print("📈 Percentile Analysis — Overall Percentage:")
for p in [25, 50, 75, 90]:
    print(f"   {p}th percentile : {np.percentile(pct_arr, p):.2f}%")

# ── 7d. Boolean masking ───────────────────────────────────────────────────────
above_avg_mask = pct_arr > np.mean(pct_arr)
print(f"\n✅ Students above class average ({np.mean(pct_arr):.2f}%): {np.sum(above_avg_mask)}")
print(f"❌ Students below class average              : {np.sum(~above_avg_mask)}")

# ── 7e. Normalize attendance (0-1) ───────────────────────────────────────────
norm_attend = (attend_arr - attend_arr.min()) / (attend_arr.max() - attend_arr.min())
df["attendance_normalized"] = np.round(norm_attend, 4)
print(f"\n📏 Normalized Attendance (0–1):")
print(f"   Min: {norm_attend.min():.4f}  |  Max: {norm_attend.max():.4f}  |  Mean: {norm_attend.mean():.4f}")


## 📊 Step 8 — Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Student Performance Dashboard", fontsize=16, fontweight="bold", y=1.01)

# ── Plot 1: Grade Distribution (Bar) ─────────────────────────────────────────
grade_counts = df["grade"].value_counts().sort_index()
colors = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336"]
axes[0,0].bar(grade_counts.index, grade_counts.values,
              color=colors[:len(grade_counts)], edgecolor="white", linewidth=1.5)
axes[0,0].set_title("Grade Distribution", fontweight="bold")
axes[0,0].set_xlabel("Grade"); axes[0,0].set_ylabel("Number of Students")
for i, v in enumerate(grade_counts.values):
    axes[0,0].text(i, v + 0.1, str(v), ha="center", fontweight="bold")

# ── Plot 2: Avg Percentage by Course (Horizontal Bar) ────────────────────────
course_pct = df.groupby("course")["percentage"].mean().sort_values()
axes[0,1].barh(course_pct.index, course_pct.values,
               color="#00BCD4", edgecolor="white", linewidth=1.5)
axes[0,1].set_title("Avg Percentage by Course", fontweight="bold")
axes[0,1].set_xlabel("Average Percentage (%)")
for i, v in enumerate(course_pct.values):
    axes[0,1].text(v + 0.3, i, f"{v:.1f}%", va="center", fontweight="bold")

# ── Plot 3: Attendance vs Percentage (Scatter) ────────────────────────────────
scatter_colors = df["grade"].map({"A+":"#4CAF50","A":"#2196F3","B":"#FF9800",
                                   "C":"#FF5722","D":"#9E9E9E"})
axes[1,0].scatter(df["attendance_percent"], df["percentage"],
                  c=scatter_colors, s=80, alpha=0.85, edgecolors="white", linewidth=0.8)
axes[1,0].set_title("Attendance vs Overall Percentage", fontweight="bold")
axes[1,0].set_xlabel("Attendance (%)"); axes[1,0].set_ylabel("Overall Percentage (%)")
# Trend line
z = np.polyfit(df["attendance_percent"], df["percentage"], 1)
p = np.poly1d(z)
x_line = np.linspace(df["attendance_percent"].min(), df["attendance_percent"].max(), 100)
axes[1,0].plot(x_line, p(x_line), "r--", linewidth=1.5, alpha=0.7, label="Trend")
axes[1,0].legend()

# ── Plot 4: Subject-wise Average (Grouped Bar) ────────────────────────────────
courses  = df["course"].unique()
x        = np.arange(len(courses))
width    = 0.25
math_avg = [df[df["course"]==c]["marks_math"].mean()    for c in courses]
sci_avg  = [df[df["course"]==c]["marks_science"].mean() for c in courses]
eng_avg  = [df[df["course"]==c]["marks_english"].mean() for c in courses]

axes[1,1].bar(x - width, math_avg,    width, label="Math",    color="#3F51B5", alpha=0.85)
axes[1,1].bar(x,          sci_avg,    width, label="Science",  color="#E91E63", alpha=0.85)
axes[1,1].bar(x + width,  eng_avg,    width, label="English",  color="#FF9800", alpha=0.85)
axes[1,1].set_title("Subject-wise Marks by Course", fontweight="bold")
axes[1,1].set_xlabel("Course"); axes[1,1].set_ylabel("Average Marks")
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(courses, rotation=15, ha="right")
axes[1,1].legend()

plt.tight_layout()
plt.savefig("dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("📸 Dashboard saved as dashboard.png")


## 🗃️ Step 9 — Advanced SQL Queries via Pandas

In [ ]:
# Query 1: Students from Hyderabad
q1 = pd.read_sql_query(
    "SELECT name, course, percentage FROM students WHERE city = 'Hyderabad' ORDER BY percentage DESC",
    conn)
print("🏙️ Students from Hyderabad:")
print(q1.to_string(index=False))

# Query 2: Top scorer per course
q2 = pd.read_sql_query("""
    SELECT course,
           name,
           ROUND((marks_math + marks_science + marks_english) / 3.0, 2) AS avg_marks
    FROM   students
    WHERE  (marks_math + marks_science + marks_english) = (
               SELECT MAX(m.marks_math + m.marks_science + m.marks_english)
               FROM   students m WHERE m.course = students.course)
    ORDER  BY avg_marks DESC
""", conn)
print("\n🥇 Top Scorer per Course:")
print(q2.to_string(index=False))


## 💾 Step 10 — Export Results to CSV

In [ ]:
output_path = "student_analysis_results.csv"
df.to_csv(output_path, index=False)
print(f"✅ Results exported to '{output_path}'  ({len(df)} rows)")

# Download the CSV in Colab
from google.colab import files
files.download(output_path)
print("📥 Download started!")


In [ ]:
conn.close()
print("🔒 Database connection closed. Practical complete!")


---
## ✅ Summary of Operations Performed

| # | Operation | Library |
|---|-----------|---------|
| 1 | Upload & load external `.sql` file | `google.colab`, `sqlite3` |
| 2 | Read SQL table into DataFrame | `pandas` |
| 3 | Added calculated columns (total marks, percentage, grade) | `pandas` |
| 4 | Filtered top students (≥85%) | `pandas` |
| 5 | GroupBy — course-wise & gender-wise aggregation | `pandas` |
| 6 | Sorting, value counts, nlargest | `pandas` |
| 7 | Descriptive statistics (mean, std, var, median) | `numpy` |
| 8 | Correlation matrix | `numpy` |
| 9 | Percentile analysis & Boolean masking | `numpy` |
| 10 | Min-max normalization | `numpy` |
| 11 | Visualizations (bar, scatter, grouped bar) | `matplotlib` |
| 12 | Advanced SQL queries via pandas | `pandas`, `sqlite3` |
| 13 | Export results to CSV | `pandas` |
